# Gold Layer - Daily Market Summary

## Purpose
Aggregate hourly market prices to **daily summary by region** for executive reporting.

## Business Question
**"What is the daily market situation by region?"**

## Output Table
`vattenfall_dev.gold.gold_daily_market_summary`

**Grain:** `report_day` × `region`

**Measures:**
* Average market price
* Maximum market price
* Total market volume
* High-price hour count

## Source
`vattenfall_dev.refined.silver_market_prices` (826 hourly records)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("✅ Setup complete")

In [0]:
# Load silver market prices
df_silver = spark.table("vattenfall_dev.refined.silver_market_prices")

print(f"✅ Loaded {df_silver.count():,} hourly market price records")
print(f"   Date range: {df_silver.agg(F.min('report_day')).first()[0]} to {df_silver.agg(F.max('report_day')).first()[0]}")
print(f"   Regions: {df_silver.select('region_normalized').distinct().count()}")

# Show sample
display(df_silver.limit(5))

In [0]:
# Calculate regional 90th percentile for high-price threshold
regional_p90 = df_silver.groupBy("region_normalized").agg(
    F.expr("percentile_approx(price_eur_mwh, 0.90)").alias("p90_threshold")
)

# Join back to main data to flag high-price hours
df_with_threshold = df_silver.join(regional_p90, "region_normalized")

# Aggregate to daily level by region
df_gold = df_with_threshold.groupBy(
    "report_day",
    F.col("region_normalized").alias("region")
).agg(
    # Average market price
    F.round(F.avg("price_eur_mwh"), 2).alias("avg_price_eur_mwh"),
    
    # Maximum market price
    F.round(F.max("price_eur_mwh"), 2).alias("max_price_eur_mwh"),
    
    # Minimum market price (useful for context)
    F.round(F.min("price_eur_mwh"), 2).alias("min_price_eur_mwh"),
    
    # Total market volume
    F.round(F.sum("volume_mwh"), 2).alias("total_volume_mwh"),
    
    # High-price hour count (hours above regional P90)
    F.sum(F.when(F.col("price_eur_mwh") >= F.col("p90_threshold"), 1).otherwise(0)).alias("high_price_hours"),
    
    # Price volatility (standard deviation)
    F.round(F.stddev("price_eur_mwh"), 2).alias("price_volatility_stddev"),
    
    # Hourly observations count
    F.count("*").alias("hourly_observations")
).orderBy("report_day", "region")

print(f"✅ Aggregated to {df_gold.count()} daily summaries")
print(f"   Grain: report_day × region")

# Show sample
display(df_gold.limit(10))

In [0]:
# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS vattenfall_dev.gold")

print("✅ Gold schema ready")
print(f"   Location: vattenfall_dev.gold")

In [0]:
# Write to Gold layer
table_name = "vattenfall_dev.gold.gold_daily_market_summary"

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Successfully wrote to {table_name}")
print(f"   Records: {spark.table(table_name).count():,}")
print(f"   Columns: {len(spark.table(table_name).columns)}")

In [0]:
# Reload for validation
df_validate = spark.table("vattenfall_dev.gold.gold_daily_market_summary")

print("=" * 70)
print("GOLD TABLE VALIDATION")
print("=" * 70)

# 1. Check grain uniqueness
print("\n1. GRAIN VALIDATION (report_day × region)")
print("-" * 70)
total_rows = df_validate.count()
unique_combos = df_validate.select("report_day", "region").distinct().count()
print(f"   Total rows: {total_rows}")
print(f"   Unique day-region combinations: {unique_combos}")
print(f"   ✅ Grain is unique: {total_rows == unique_combos}")

# 2. Check for nulls in key columns
print("\n2. NULL CHECK")
print("-" * 70)
for col in ["report_day", "region", "avg_price_eur_mwh", "total_volume_mwh"]:
    null_count = df_validate.filter(F.col(col).isNull()).count()
    status = "✅ PASS" if null_count == 0 else f"❌ {null_count} nulls"
    print(f"   {col:.<30} {status}")

# 3. Date coverage
print("\n3. DATE COVERAGE")
print("-" * 70)
date_stats = df_validate.agg(
    F.min("report_day").alias("start_date"),
    F.max("report_day").alias("end_date"),
    F.countDistinct("report_day").alias("unique_days")
).first()
print(f"   Start date: {date_stats['start_date']}")
print(f"   End date: {date_stats['end_date']}")
print(f"   Unique days: {date_stats['unique_days']}")

# 4. Regional coverage
print("\n4. REGIONAL COVERAGE")
print("-" * 70)
df_validate.groupBy("region").agg(
    F.count("*").alias("days"),
    F.round(F.avg("avg_price_eur_mwh"), 2).alias("avg_price")
).orderBy("region").show()

# 5. Top 5 highest price days
print("\n5. TOP 5 HIGHEST AVERAGE PRICE DAYS")
print("-" * 70)
df_validate.orderBy(F.col("avg_price_eur_mwh").desc()).limit(5).select(
    "report_day", "region", "avg_price_eur_mwh", "max_price_eur_mwh", 
    "high_price_hours", "price_volatility_stddev"
).show(truncate=False)

print("\n" + "=" * 70)
print("✅ Gold table validation complete")
print("=" * 70)

## ✅ Gold Table Complete

### Output
**Table:** `vattenfall_dev.gold.gold_daily_market_summary`

**Grain:** `report_day` × `region`

**Columns:**
* `report_day` - Date
* `region` - Country code (DK, FI, NO, SE)
* `avg_price_eur_mwh` - Average daily price
* `max_price_eur_mwh` - Maximum hourly price
* `min_price_eur_mwh` - Minimum hourly price
* `total_volume_mwh` - Total daily volume
* `high_price_hours` - Count of hours above regional P90
* `price_volatility_stddev` - Price standard deviation
* `hourly_observations` - Number of hourly records

### Business Question Answered
**"What is the daily market situation by region?"**

This table provides:
* ✅ Daily price levels (avg, max, min)
* ✅ Market activity (volume)
* ✅ Price volatility (stddev)
* ✅ High-price stress indicator (hours above P90)

### Usage Example
```sql
SELECT 
  report_day,
  region,
  avg_price_eur_mwh,
  high_price_hours
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE report_day >= '2024-01-01'
ORDER BY avg_price_eur_mwh DESC
LIMIT 10;
```

In [0]:
%sql
SELECT 
  report_day,
  region,
  avg_price_eur_mwh,
  high_price_hours
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE report_day >= '2024-01-01'
ORDER BY avg_price_eur_mwh DESC
LIMIT 10;

---

## Business Question: "What is the daily market situation by region?"

### Executive Summary (January 1-15, 2024)

Analysis of **60 daily market summaries** across 4 Nordic regions reveals distinct pricing profiles and market stress patterns:

### Regional Market Profiles

**🔴 Finland - Premium Market (48.62 EUR/MWh average)**
* **Highest cost region** - 9% premium over DK/NO
* **Dominates expensive days:** 6 of top 10 highest-price days
* **Peak stress:** January 9 at **56.66 EUR/MWh** (reaching 66.38 EUR/MWh hourly max)
* **Volatility concern:** January 12 showed extreme swings (14.55 stddev) despite moderate average
* **Pattern:** Consistently elevated pricing suggests structural capacity constraints or demand pressures

**🟡 Sweden - Moderate with Stress Episodes (44.98 EUR/MWh average)**
* Mid-range pricing between Finland (high) and DK/NO (low)
* **Notable stress periods:** January 10 and 15 each had **4 high-price hours** (above regional P90)
* Appears to bridge Nordic price dynamics

**🟢 Denmark & Norway - Lowest Cost Markets (44.40 EUR/MWh average - tied)**
* **Most affordable** regional pricing
* Norway experienced **one extreme spike:** January 7 at 55.27 EUR/MWh (second-highest overall)
* Generally stable pricing environment
* Denmark shows no extreme outliers in top 10
